# 00 · Vertical slice: network revenue in one pass

Minimal end-to-end run of the Interchange Pricing Simulator: synthetic data → parquet →
P&L by actor → network revenue.

> **Synthetic data.** Every figure below follows from the assumptions in `config/base.yaml`
> and `docs/assumptions.md`. The deliverable is the framework, not the numbers.

In [1]:
import time

import duckdb
import polars as pl

from ips.data_gen.simple_generator import generate_all, write_parquet
from ips.economics.simple_pnl import compute_pnl, interchange_table_frame, pnl_by
from ips.utils.config import load_config

start = time.perf_counter()
pl.Config.set_thousands_separator(",")
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)

cfg = load_config()
print(f"seed={cfg.seed}  period={cfg.dates.start} to {cfg.dates.end}")

seed=20260910  period=2025-01-01 to 2025-12-31


## 1. Generate the synthetic network

In [2]:
data = generate_all(cfg)
raw_dir = cfg.resolve_path(cfg.output.raw_dir)
paths = write_parquet(data, raw_dir)
pl.DataFrame(
    {"table": [p.stem for p in paths], "rows": [data.tables()[p.stem].height for p in paths]}
)

table,rows
str,i64
"""issuers""",6
"""acquirers""",4
"""merchants""","2,000"
"""cardholders""","20,000"
"""transactions""","100,000"


## 2. Minimal pipeline: read the parquet back and cross-check it with DuckDB

In [3]:
txn_path = raw_dir / "transactions.parquet"
transactions = pl.read_parquet(txn_path)

n_txns, gdv = duckdb.sql(
    f"select count(*), sum(amount_cop) from read_parquet('{txn_path.as_posix()}')"
).fetchone()
assert n_txns == transactions.height
assert gdv == transactions["amount_cop"].sum()
transactions.head()

txn_id,txn_date,cardholder_id,merchant_id,issuer_id,acquirer_id,product,mcc,amount_cop,on_us
u32,date,u32,u32,str,str,str,str,i64,bool
0,2025-01-01,"15,417",785,"""ISS_BANK_C""","""ACQ_AGG_X""","""credit_standard""","""5311""","716,771",false
1,2025-01-01,"7,563","1,013","""ISS_BANK_B""","""ACQ_BANK_A""","""debit""","""5812""","97,556",false
2,2025-01-01,"17,821","1,550","""ISS_BANK_B""","""ACQ_AGG_X""","""credit_standard""","""5411""","54,447",false
3,2025-01-01,"18,145",642,"""ISS_BANK_A""","""ACQ_BANK_A""","""credit_premium""","""4511""","2,224,176",true
4,2025-01-01,"6,584","1,353","""ISS_BANK_A""","""ACQ_AGG_X""","""credit_standard""","""5812""","42,193",false


## 3. P&L by actor

Every peso of MDR paid by merchants ends up with exactly one receiver: the issuer
(interchange), the network (scheme fee) or the acquirer (the residual margin).

In [4]:
table = interchange_table_frame(cfg.pricing)
pnl = compute_pnl(transactions, table, cfg.pricing)
totals = pnl.totals()


def whole_cop(*columns: str) -> pl.Expr:
    return pl.col(*columns).round(0).cast(pl.Int64)


by_type = (
    pnl.by_actor.group_by("actor_type", maintain_order=True)
    .agg(pl.col("revenue_cop").sum())
    .with_columns(
        (100 * pl.col("revenue_cop") / totals["mdr_cop"]).round(1).alias("share_of_mdr_%")
    )
)
# Conservation: issuers + network + acquirers must add up to what merchants paid.
assert abs(by_type["revenue_cop"].sum() - totals["mdr_cop"]) <= 1e-9 * totals["mdr_cop"]
by_type.with_columns(whole_cop("revenue_cop"))

actor_type,revenue_cop,share_of_mdr_%
str,i64,f64
"""issuer""","252,625,422",63.8
"""acquirer""","118,004,137",29.8
"""network""","25,537,989",6.4


Detail by institution:

In [5]:
pnl.by_actor.with_columns(whole_cop("revenue_cop", "gdv_cop"))

actor_type,actor_id,revenue_cop,gdv_cop,n_txns
str,str,i64,i64,i64
"""issuer""","""ISS_BANK_A""","82,550,837","6,249,601,510","31,541"
"""issuer""","""ISS_BANK_B""","62,182,423","4,667,090,042","24,214"
"""issuer""","""ISS_BANK_C""","41,946,474","3,112,145,078","15,774"
"""issuer""","""ISS_BANK_D""","27,614,523","2,030,739,325","10,296"
"""issuer""","""ISS_FINTECH_E""","25,260,177","2,365,703,039","12,061"
"""issuer""","""ISS_FINTECH_F""","13,070,989","1,219,327,829","6,114"
"""acquirer""","""ACQ_AGG_X""","26,610,053","4,381,195,149","22,939"
"""acquirer""","""ACQ_AGG_Y""","22,599,068","3,765,404,834","21,086"
"""acquirer""","""ACQ_BANK_A""","39,270,921","6,518,715,726","33,108"


## 4. Card mix drives the acquirer's margin

Under blended pricing the merchant pays the same rate whatever card it is presented, so the
acquirer absorbs the difference: premium cards carry more interchange than the blended rate
covers.

In [6]:
rates = ["effective_interchange_rate", "effective_mdr_rate", "acquirer_margin_rate"]
pnl_by(pnl, ["product"]).select(
    "product",
    "n_txns",
    whole_cop("gdv_cop"),
    *[(100 * pl.col(r)).round(2).alias(r.removesuffix("_rate") + "_%") for r in rates],
)

product,n_txns,gdv_cop,effective_interchange_%,effective_mdr_%,acquirer_margin_%
str,i64,i64,f64,f64,f64
"""credit_premium""","13,465","2,618,155,915",2.14,2.02,-0.25
"""credit_standard""","27,346","5,360,312,368",1.72,2.01,0.16
"""debit""","59,189","11,666,138,540",0.89,2.02,0.99


## 5. On-us share

Transactions whose issuer and the merchant's acquirer belong to the same bank group.
Acquirers are assigned to merchants independently of the cardholders' issuers.

In [7]:
pnl_by(pnl, ["on_us"]).select(
    "on_us",
    "n_txns",
    (100 * pl.col("n_txns") / pl.col("n_txns").sum()).round(1).alias("share_of_txns_%"),
    whole_cop("gdv_cop"),
)

on_us,n_txns,share_of_txns_%,gdv_cop
bool,i64,f64,i64
false,"84,029",84.0,"16,421,109,803"
true,"15,971",16.0,"3,223,497,020"


## 6. Headline

In [8]:
elapsed = time.perf_counter() - start
print(
    f"Network revenue (scheme fees) = {pnl.network_revenue:,.0f} COP | "
    f"net revenue yield = {totals['net_revenue_yield_bps']:.1f} bps "
    f"on GDV of {totals['gdv_cop']:,.0f} COP"
)
print(f"End-to-end runtime: {elapsed:.1f} s")

Network revenue (scheme fees) = 25,537,989 COP | net revenue yield = 13.0 bps on GDV of 19,644,606,823 COP
End-to-end runtime: 0.2 s


## Limits of this slice

- The yield equals the scheme fee rate (0.13%) by construction: with a single percentage
  scheme fee and no volume response, network revenue is just rate × GDV. It becomes
  informative once cross-border fees, authorization fees and acceptance elasticity exist
  (Features 1, 3, 5 and 6).
- Uniform dates, no channel, no cross-border, no fraud, blended pricing only and no issuer
  costs (rewards, fraud, funding). See `docs/assumptions.md`.
- Next: Feature 1 replaces this generator with a 5M+ transaction affinity process.